# QLoRA Fine-Tuning: Legal Demand Assistant

Fine-tunes **Qwen3.5-9B** on a 50-example legal instruction dataset using QLoRA (4-bit quantization + LoRA adapters).

**Requirements:** Colab Pro with A100 GPU (~15-20 GB VRAM)

| Parameter | Value |
|---|---|
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Epochs | 3 |
| Learning rate | 2e-4 |
| Effective batch size | 8 (2 × 4 grad accum) |
| Max sequence length | 1024 |

## Section 0: Install Dependencies

In [4]:
import torch
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [6]:
!pip install transformers>=4.46.0 peft>=0.13.0 bitsandbytes>=0.44.0 trl>=0.12.0 datasets accelerate scikit-learn

In [7]:
!pip install flash-attn --no-build-isolation

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


## Section 1: Imports and Configuration

In [2]:
import json
import os

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# ── Hyperparameters ──────────────────────────────────────────────────────────

MODEL_ID = "Qwen/Qwen3.5-9B"
OUTPUT_DIR = "./qlora-legal-demand-adapter"
DATA_PATH = "./data/instruction_dataset.json"  # Update if your repo is cloned elsewhere

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 1024
WEIGHT_DECAY = 0.01

SYSTEM_PROMPT = (
    "You are a legal demand assistant. You help draft demand letters, "
    "identify legal claims, extract elements from demand letters, "
    "evaluate letter quality, and recommend remedies."
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Section 2: Upload / Clone Dataset

Upload `instruction_dataset.json` or clone the repo. Adjust `DATA_PATH` above if needed.

In [3]:
# Option A: Clone the repo (uncomment and update URL)
!git clone https://github.com/BigDataAnalytics-CS5542/Lab_8.git
DATA_PATH = "./Lab_8/data/instruction_dataset.json"

# Option B: Upload the file via Colab UI
# from google.colab import files
# uploaded = files.upload()  # upload instruction_dataset.json
# DATA_PATH = "./instruction_dataset.json"

# Verify the file exists
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH} — update DATA_PATH"
print(f"Dataset found: {DATA_PATH}")

fatal: destination path 'Lab_8' already exists and is not an empty directory.
Dataset found: ./Lab_8/data/instruction_dataset.json


## Section 3: Load and Format Dataset

In [4]:
with open(DATA_PATH) as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} examples")
print(f"Task types: {set(ex['task_type'] for ex in raw_data)}")

# Format into ChatML messages
formatted = []
for example in raw_data:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example["output"]},
    ]
    formatted.append({
        "messages": messages,
        "task_type": example["task_type"],
    })

dataset = Dataset.from_list(formatted)

# Stratified 90/10 split (45 train / 5 val) by task_type
task_types = dataset["task_type"]
indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices, test_size=5, random_state=42, stratify=task_types
)

train_ds = dataset.select(train_idx).remove_columns(["task_type"])
val_ds = dataset.select(val_idx).remove_columns(["task_type"])

print(f"Train: {len(train_ds)} examples, Val: {len(val_ds)} examples")

Loaded 50 examples
Task types: {'evaluate_letter', 'extract_elements', 'draft_demand_letter', 'identify_claim', 'recommend_remedy'}
Train: 45 examples, Val: 5 examples


## Section 4: Load Model and Tokenizer (4-bit Quantization)

In [14]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 122.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

print(f"Model loaded: {MODEL_ID}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3.5-9B
Model memory: 7.53 GB


## Section 5: Configure LoRA and Training

In [11]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
)

# Note: In some versions of TRL, max_seq_length is a direct argument to SFTTrainer
# or part of SFTConfig. We will use SFTConfig but ensure the naming is strictly per the latest API.
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    report_to="none",
    seed=42,
)

print("LoRA config:", peft_config)
print(f"\nTraining: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

LoRA config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'gate_proj', 'v_proj', 'q_proj', 'k_proj', 'o_proj', 'up_proj', 'down_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

Training: 3 epochs, lr=0.0002, effective batch=8


## Section 6: Train

In [16]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Starting training...")

result = trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Trainable params: 29,097,984
Starting training...


Epoch,Training Loss,Validation Loss
1,1.429862,0.898238
2,0.726433,0.743085
3,0.551868,0.724605


In [17]:
# Print training results
print("── Training Results ──")
print(f"  Total steps:    {result.global_step}")
print(f"  Training loss:  {result.training_loss:.4f}")
for key, value in result.metrics.items():
    print(f"  {key}: {value}")

── Training Results ──
  Total steps:    18
  Training loss:  0.8512
  train_runtime: 162.6423
  train_samples_per_second: 0.83
  train_steps_per_second: 0.111
  total_flos: 1324868950056960.0
  train_loss: 0.8512331446011862


## Section 7: Save Adapter Weights

In [18]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

# List saved files
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s} {size / 1e6:.1f} MB")

Adapter saved to ./qlora-legal-demand-adapter
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                116.4 MB
  chat_template.jinja                      0.0 MB
  checkpoint-12                            0.0 MB
  checkpoint-18                            0.0 MB
  checkpoint-6                             0.0 MB
  tokenizer.json                           20.0 MB
  tokenizer_config.json                    0.0 MB
  training_args.bin                        0.0 MB


## Section 8: Inference Test

Reload the base model + adapter and test with sample prompts to verify the fine-tuning worked.

In [19]:
# Free training memory before inference
del trainer, model
torch.cuda.empty_cache()

# Reload base model + adapter
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config_inf,
    attn_implementation="eager",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
print("Model + adapter loaded for inference")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model + adapter loaded for inference


In [20]:
test_prompts = [
    {
        "task": "Demand Letter",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Draft a legal demand letter based on the scenario.\n\n"
                "A dog walker lost a client's dog due to negligence. The dog was "
                "found two days later, but the owner incurred $1,200 in search and "
                "veterinary costs."
            )},
        ],
    },
    {
        "task": "Claim Identification",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Identify the strongest legal claim based on the scenario.\n\n"
                "A mechanic charged $800 for car repairs that were never actually performed."
            )},
        ],
    },
    {
        "task": "Element Extraction",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Extract the key legal elements from this demand letter.\n\n"
                "Dear QuickFix Plumbing, I represent Maria Santos regarding "
                "water damage totaling $3,600 caused by your faulty pipe installation "
                "on February 15, 2026. Please remit payment within 14 days."
            )},
        ],
    },
]

for prompt in test_prompts:
    text = tokenizer.apply_chat_template(
        prompt["messages"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print(f"\n{'='*60}")
    print(f"Task: {prompt['task']}")
    print(f"{'='*60}")
    print(response.strip())

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



Task: Demand Letter
Dear [Walker Name],

I represent [Owner Name] regarding the loss of my client’s dog during your care. Despite your agreement to look after the dog, the animal was not returned promptly, and my client incurred $1,200 in search and veterinary expenses.

My client demands reimbursement of $1,200 within 7 days of receipt of this letter. If payment is not made by that date, my client will consider other legal remedies to recover the losses.

Sincerely,
[Sender Name]


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



Task: Claim Identification
{
  "claim_type": "fraud_or_misrepresentation",
  "reasoning": "The mechanic charged for services that were never performed, which may constitute fraud or unjust enrichment."
}

Task: Element Extraction
{
  "parties": {
    "claimant": "Maria Santos",
    "recipient": "QuickFix Plumbing"
  },
  "claim_type": "breach_of_contract",
  "damages": "$3,600",
  "deadline": "14 days",
  "remedy_requested": "Payment of $3,600"
}


## Section 9: Download Adapter (Optional)

Download the adapter weights to your local machine for use in the FastAPI pipeline.

In [21]:
# Zip and download the adapter
!zip -r qlora-legal-demand-adapter.zip {OUTPUT_DIR}

# Uncomment to download via Colab UI
from google.colab import files
files.download("qlora-legal-demand-adapter.zip")

  adding: qlora-legal-demand-adapter/ (stored 0%)
  adding: qlora-legal-demand-adapter/checkpoint-18/ (stored 0%)
  adding: qlora-legal-demand-adapter/checkpoint-18/optimizer.pt (deflated 23%)
  adding: qlora-legal-demand-adapter/checkpoint-18/chat_template.jinja (deflated 76%)
  adding: qlora-legal-demand-adapter/checkpoint-18/tokenizer.json (deflated 80%)
  adding: qlora-legal-demand-adapter/checkpoint-18/adapter_model.safetensors (deflated 22%)
  adding: qlora-legal-demand-adapter/checkpoint-18/tokenizer_config.json (deflated 65%)
  adding: qlora-legal-demand-adapter/checkpoint-18/scheduler.pt (deflated 62%)
  adding: qlora-legal-demand-adapter/checkpoint-18/adapter_config.json (deflated 59%)
  adding: qlora-legal-demand-adapter/checkpoint-18/training_args.bin (deflated 52%)
  adding: qlora-legal-demand-adapter/checkpoint-18/README.md (deflated 65%)
  adding: qlora-legal-demand-adapter/checkpoint-18/rng_state.pth (deflated 26%)
  adding: qlora-legal-demand-adapter/checkpoint-18/trai

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>